# Setup Directories Table

This notebook connects to boilest.db and creates the directories table if it doesn't exist.

In [4]:
import sqlite3
import os
from datetime import datetime

## Setup Directories Table Function

This function handles the complete setup process: connecting to the database, creating the table, and verifying the schema.

In [11]:
def setup_directories_table():
    """
    Connect to boilest.db and create the directories table if it doesn't exist.
    
    Schema:
    - guid: TEXT PRIMARY KEY
    - path: TEXT NOT NULL
    - added_at: TEXT NOT NULL
    """
    # Determine database path based on environment
    if os.environ.get("Role") == "Manager":
        db_path = "/boil/app/boilest.db"
    else:
        db_path = 'boilest.db'
    
    print(f"Database path: {db_path}")
    
    # Connect to the database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    print("Connected to database successfully")
    
    try:
        # Check if directories table already exists
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='directories'")
        table_exists = cursor.fetchone() is not None
        
        if table_exists:
            print("Directories table already exists")
        else:
            # Create directories table if it doesn't exist
            create_table_sql = """
            CREATE TABLE IF NOT EXISTS directories (
                guid TEXT PRIMARY KEY,
                path TEXT NOT NULL,
                added_at TEXT NOT NULL,
                ffmpeg_video TEXT,
                ffmpeg_audio TEXT,
                ffmpeg_subtitle TEXT
            )
            """
            
            cursor.execute(create_table_sql)
            conn.commit()
            print("Directories table created successfully")
        
        # Verify the table was created by checking its schema
        cursor.execute("PRAGMA table_info(directories)")
        columns = cursor.fetchall()
        
        print("\nTable schema:")
        for col in columns:
            print(f"  {col[1]} ({col[2]})")
    
    finally:
        # Close the database connection
        conn.close()
        print("\nDatabase connection closed")

setup_directories_table()

Database path: boilest.db
Connected to database successfully
Directories table already exists

Table schema:
  guid (TEXT)
  path (TEXT)
  added_at (TEXT)
  ffmpeg_video (TEXT)
  ffmpeg_audio (TEXT)
  ffmpeg_subtitle (TEXT)

Database connection closed


In [3]:
def add_directory_codec_columns():
    """
    Add codec columns to the directories table if they are missing.
    """
    if os.environ.get("Role") == "Manager":
        db_path = "/boil/app/boilest.db"
    else:
        db_path = "boilest.db"

    print(f"Database path: {db_path}")

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        cursor.execute("PRAGMA table_info(directories)")
        existing_columns = {row[1] for row in cursor.fetchall()}

        columns_to_add = [
            ("video_codec", "TEXT"),
            ("audio_codec", "TEXT"),
            ("subtitle_codec", "TEXT"),
        ]

        for column_name, column_type in columns_to_add:
            if column_name not in existing_columns:
                cursor.execute(
                    f"ALTER TABLE directories ADD COLUMN {column_name} {column_type}"
                )

        conn.commit()

        cursor.execute("PRAGMA table_info(directories)")
        columns = cursor.fetchall()

        print("\nUpdated table schema:")
        for col in columns:
            print(f"  {col[1]} ({col[2]})")

    finally:
        conn.close()
        print("\nDatabase connection closed")

add_directory_codec_columns()

Database path: boilest.db

Updated table schema:
  guid (TEXT)
  path (TEXT)
  added_at (TEXT)
  ffmpeg_video (TEXT)
  ffmpeg_audio (TEXT)
  ffmpeg_subtitle (TEXT)
  video_codec (TEXT)
  audio_codec (TEXT)
  subtitle_codec (TEXT)

Database connection closed


## Add Row to Directories Function

Function to insert a new directory record into the directories table.

In [10]:
def add_directory(path):
    """
    Add a new directory record to the directories table.
    
    Args:
        path (str): Directory path
    
    Returns:
        str: GUID of the added directory if successful, None otherwise
    """
    from uuid import uuid4
    
    # Generate guid and timestamp
    guid = str(uuid4())
    added_at = datetime.now().isoformat()
    
    # Determine database path based on environment
    if os.environ.get("Role") == "Manager":
        db_path = "/boil/app/boilest.db"
    else:
        db_path = 'boilest.db'
    
    try:
        # Connect to the database
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Check if directory path already exists
        check_sql = "SELECT guid FROM directories WHERE path = ?"
        cursor.execute(check_sql, (path,))
        existing = cursor.fetchone()
        
        if existing:
            print(f"Directory already exists: path={path}, guid={existing[0]}")
            conn.close()
            return existing[0]
        
        # Insert the directory record
        insert_sql = """
        INSERT INTO directories (guid, path, added_at)
        VALUES (?, ?, ?)
        """
        
        cursor.execute(insert_sql, (guid, path, added_at))
        conn.commit()
        
        print(f"Directory added successfully: guid={guid}, path={path}")
        return guid
    
    except sqlite3.IntegrityError as e:
        print(f"Error: Failed to add directory. {e}")
        return None
    
    except Exception as e:
        print(f"Unexpected error: {e}")
        return None
    
    finally:
        conn.close()

add_directory('/tv')
add_directory('/tv_favorites')
add_directory('/movies')
add_directory('/movies_favorites')
add_directory('/anime')
add_directory('/anime_favorites')
add_directory('/home_movies')
add_directory('/youtube_archive')












Directory already exists: path=/tv, guid=606a4bb5-5488-4fe5-ad1f-313a452dcf5e
Directory already exists: path=/tv_favorites, guid=a1e68ef3-33f0-45ae-ac56-2107cd2ee221
Directory already exists: path=/movies, guid=9f220361-f519-4daf-8a39-6acc7de2f8c4
Directory already exists: path=/movies_favorites, guid=cb07da93-8d17-44b0-b392-dad4d88a1d4d
Directory already exists: path=/anime, guid=6d65f65e-7df9-4984-9ecd-4ba7cd46fced
Directory already exists: path=/anime_favorites, guid=536d5cbe-2ad2-4519-9840-2be48cfd4b51
Directory already exists: path=/home_movies, guid=2a6840f2-c421-40ee-a2ef-5424f37ac581
Directory already exists: path=/youtube_archive, guid=6c0ad476-f985-4a53-b9d9-b8ab08c1cef6


'6c0ad476-f985-4a53-b9d9-b8ab08c1cef6'

In [5]:
def add_worker_column():
    """
    Add worker column to the queue table if it doesn't exist.
    """

    db_path = "boilest.db"

    print(f"Database path: {db_path}")

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        cursor.execute("PRAGMA table_info(queue)")
        existing_columns = {row[1] for row in cursor.fetchall()}

        if "worker" not in existing_columns:
            cursor.execute(
                "ALTER TABLE queue ADD COLUMN worker TEXT"
            )
            print("Worker column added successfully")
        else:
            print("Worker column already exists")

        conn.commit()

        cursor.execute("PRAGMA table_info(queue)")
        columns = cursor.fetchall()

        print("\nUpdated table schema:")
        for col in columns:
            print(f"  {col[1]} ({col[2]})")

    finally:
        conn.close()
        print("\nDatabase connection closed")

# Uncomment to execute:
add_worker_column()

Database path: boilest.db
Worker column added successfully

Updated table schema:
  directory_guid (TEXT)
  file_guid (TEXT)
  directory_path (TEXT)
  input_file_name (TEXT)
  output_file_name (TEXT)
  before_file_size (INTEGER)
  after_file_size (INTEGER)
  ffmpeg_string (TEXT)
  datetime_added (TEXT)
  datetime_pulled (TEXT)
  datetime_encoded (TEXT)
  status (TEXT)
  worker (TEXT)

Database connection closed
